# AIBO（aibo_v8 リポジトリ）— Google Colab · Nunchaku 経路

- **RunPod 用ファイル（Dockerfile / `runpod_*.py` 等）はローカルリポ内にありますが、このノートでは触りません。**
- 実行は **`02_colab_setup.py` の `ColabBootstrap` + `07_main.run()`**（`01_config.RuntimeStrategy` に従う Colab 経路）です。
- **Colab の A100 40GB** では `01_config.py` の `RuntimeStrategy.detect()` が通常 **`a100_40gb_nunchaku`** を選びます（VRAM ≥ 38GB かつ GPU 名が A6000 系ルールに当たらない場合）。**A100 80GB** では `a100_80gb_bf16` になります。
- **`!python 02_colab_setup.py` 単体**はリポ内で **診断・wheel 候補表示のみ**（`bootstrap.run()` 未実行）なので、下のセルでは **`ColabBootstrap(...).run()` を必ず続けて実行**します。
- **`pip install nunchaku` は非推奨**（PyPI の別物パッケージと衝突することがあります）。**公式 wheel は `ColabBootstrap` が入れます。**

In [ ]:
!git clone https://github.com/miya390831-a11y/aibo_v8.git
%cd aibo_v8

In [ ]:
!nvidia-smi

In [ ]:
# 依存・Drive マウント・公式 Nunchaku wheel・PuLID 取得など（Colab 本番セットアップ）
# 参考: !pip install -q nunchaku  ← PyPI 偽パッケージのリスクがあるため省略推奨

import subprocess
import sys
from importlib import import_module

# リポジトリ同梱の診断（wheel URL 候補の表示など）。フルセットアップは続く bootstrap.run()。
subprocess.run([sys.executable, "02_colab_setup.py"], cwd=".", check=False)

cfg_mod = import_module("01_config")
setup_mod = import_module("02_colab_setup")
sys_cfg = cfg_mod.SystemConfig()
bootstrap = setup_mod.ColabBootstrap(sys_cfg)
bootstrap.run()

In [ ]:
from google.colab import userdata
import os

_hf = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = _hf
os.environ["HUGGINGFACE_HUB_TOKEN"] = _hf

In [ ]:
# 全 Phase（Pipeline build · Gradio UI 起動など）
# 注: !python 07_main.py はリポジトリ内の「軽量診断」のみ。
from importlib import import_module

import_module("07_main").run()

## 確認: 戦略が `a100_40gb_nunchaku` か（A100 40GB 想定）
セルがまだ GPU ランタイム上であれば、次を実行してください。**80GB では `a100_80gb_bf16` が正しい挙動**です。

In [ ]:
from importlib import import_module

r = import_module("01_config").RuntimeStrategy.detect()
print("strategy:", r.kind.value)
print("vram_gb:", round(r.vram_gb, 2), "gpu:", r.gpu_name)
print("nunchaku_available:", r.nunchaku_available)